# Vegetable Image Classification Using CNN
## 1. Phase 1 setup
Attach the [Vegetable Image Dataset](https://www.kaggle.com/datasets/misrakahmed/vegetable-image-dataset) as Kaggle input and run cells in order. This notebook covers setup and class-distribution EDA only. All counts come from observed files. The first code cell imports all Phase 1 libraries. The fixed seed supports reproducibility; counting is deterministic.

In [ ]:
import random
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
random.seed(SEED)

## 2. Discover and validate Kaggle paths
Automatically locate the dataset's train, test and validation folders under Kaggle Input, even if the mounted dataset has an extra nesting level. The code reports the detected root and fails clearly when none or multiple are found.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
SPLITS = ("train", "test", "validation")

def discover_dataset_root(input_root=INPUT_ROOT, max_depth=4):
    """Find a mounted folder containing train, test and validation directories."""
    if not input_root.is_dir():
        raise FileNotFoundError(f"Kaggle input directory is unavailable: {input_root}")

    preferred = input_root / "vegetable-image-dataset" / "Vegetable Images"
    if all((preferred / split).is_dir() for split in SPLITS):
        return preferred

    candidates = []
    pending = [(input_root, 0)]
    while pending:
        folder, depth = pending.pop()
        child_dirs = sorted((p for p in folder.iterdir() if p.is_dir()), key=str)
        if all((folder / split).is_dir() for split in SPLITS):
            candidates.append(folder)
            continue  # Do not inspect class folders or image files.
        if depth < max_depth:
            pending.extend((child, depth + 1) for child in child_dirs)

    if not candidates:
        mounted = sorted(p.name for p in input_root.iterdir())
        raise FileNotFoundError(
            "Could not find a folder containing train, test and validation under "
            f"{input_root}. Mounted inputs: {mounted}. "
            "Expand the Kaggle Input tree and confirm the dataset is attached and extracted."
        )
    if len(candidates) > 1:
        raise RuntimeError(
            "Multiple dataset roots were found. Choose one explicitly: "
            + ", ".join(str(path) for path in sorted(candidates))
        )
    return candidates[0]

DATASET_ROOT = discover_dataset_root()
TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"
VALIDATION_DIR = DATASET_ROOT / "validation"
SPLIT_PATHS = {"Train": TRAIN_DIR, "Test": TEST_DIR, "Validation": VALIDATION_DIR}

for label, path in SPLIT_PATHS.items():
    if not path.is_dir():
        raise NotADirectoryError(f"{label} directory is missing: {path}")

print(f"Detected dataset root: {DATASET_ROOT}")
for label, path in SPLIT_PATHS.items():
    print(f"{label}: {path}")


## 3. Reusable image-counting function
Count supported image filenames directly inside each class directory, case-insensitively. Preserve empty classes; ignore non-image files, split-level files, and nested directories. This measures file counts without decoding or verifying images.

In [ ]:
def count_images(split_dir):
    """Count supported image files immediately inside each class directory."""
    split_dir = Path(split_dir)
    if not split_dir.exists():
        raise FileNotFoundError(f'Split folder does not exist: {split_dir}')
    if not split_dir.is_dir():
        raise NotADirectoryError(f'Split path is not a directory: {split_dir}')
    extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tif', '.tiff'}
    class_dirs = sorted(path for path in split_dir.iterdir() if path.is_dir())
    if not class_dirs:
        raise ValueError(f'No class folders found in: {split_dir}')
    return {
        class_dir.name: sum(
            1 for path in class_dir.iterdir()
            if path.is_file() and path.suffix.lower() in extensions
        )
        for class_dir in class_dirs
    }

## 4. Count splits independently and build three DataFrames
Independent counts preserve differences in class membership across train, test, and validation.

In [ ]:
train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)
validation_counts = count_images(VALIDATION_DIR)
train_df = pd.DataFrame(list(train_counts.items()), columns=['Vegetable', 'Train Count'])
test_df = pd.DataFrame(list(test_counts.items()), columns=['Vegetable', 'Test Count'])
validation_df = pd.DataFrame(list(validation_counts.items()), columns=['Vegetable', 'Validation Count'])
for label, frame in [('Train', train_df), ('Test', test_df), ('Validation', validation_df)]:
    print(f'{label} image counts')
    display(frame)

## 5. Merge counts and report totals
Outer joins retain the union of classes. Missing class folders receive zero counts and an explicit warning. The detected class count is the union across all splits.

In [ ]:
count_columns = ['Train Count', 'Test Count', 'Validation Count']
counts_df = (
    train_df.merge(test_df, on='Vegetable', how='outer', validate='one_to_one')
    .merge(validation_df, on='Vegetable', how='outer', validate='one_to_one')
    .sort_values('Vegetable').reset_index(drop=True)
)
counts_df[count_columns] = counts_df[count_columns].fillna(0).astype('int64')
counts_df = counts_df[['Vegetable', *count_columns]]
display(counts_df)
all_classes = set(train_counts) | set(test_counts) | set(validation_counts)
print(f'Detected class count (union): {len(all_classes)}')
for label, observed in [('Train', train_counts), ('Test', test_counts), ('Validation', validation_counts)]:
    print(f'{label}: {len(observed)} class folders; {sum(observed.values())} images')
    missing = sorted(all_classes - set(observed))
    if missing:
        print(f'WARNING: {label} is missing class folders: {missing}')
print(f'Total images across all splits: {int(counts_df[count_columns].to_numpy().sum())}')

## 6. Plot class distribution
Grouped bars compare splits for each vegetable. The vertical axis measures image files and starts at zero; rotated labels improve readability.

In [ ]:
ax = counts_df.set_index('Vegetable')[count_columns].plot(kind='bar', figsize=(14, 6), width=0.8)
ax.set_title('Vegetable image counts by class and dataset split')
ax.set_xlabel('Vegetable')
ax.set_ylabel('Number of images')
ax.set_ylim(bottom=0)
ax.legend(title='Dataset split', labels=['Train', 'Test', 'Validation'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Explain balance using real counts
Balance is assessed across classes within each split; the splits need not have equal sizes. The report uses observed totals, minima, maxima, and ratios. Equal counts mean exactly balanced; unequal counts are reported without an arbitrary imbalance threshold. Zero-count classes prevent a finite maximum/minimum ratio.

In [ ]:
for label, column in zip(['Train', 'Test', 'Validation'], count_columns):
    values = counts_df[column]
    minimum, maximum = int(values.min()), int(values.max())
    total = int(values.sum())
    print(f'\n{label}: {total} images across {len(values)} union classes; minimum = {minimum}, maximum = {maximum}.')
    if total == 0:
        print('No supported images were found; image balance cannot be assessed.')
    elif minimum == maximum:
        print(f'Exactly balanced: every class has {minimum} images.')
    else:
        print('Class counts differ in this split.')
        print('Minimum-count classes:', ', '.join(counts_df.loc[values == minimum, 'Vegetable']))
        print('Maximum-count classes:', ', '.join(counts_df.loc[values == maximum, 'Vegetable']))
        if minimum > 0:
            print(f'Maximum/minimum count ratio: {maximum / minimum:.3f}')
        else:
            print('At least one class has zero images; maximum/minimum ratio is undefined.')

## 8. Phase 1 stopping point
Review the tables, warnings, chart, and balance report after running on Kaggle. No dataset counts are prefilled. Filename counts do not establish image integrity, label accuracy, duplicates, or split leakage. A separately authorized next phase can investigate these issues and plan preprocessing before CNN development. This notebook stops at setup and EDA.